# Day 13, Step 1 — Loop Engineering: three ways an act/verify loop can end

**Problem:** `example.py` already shows a loop (`act` → `verify` → retry) stopped by a single harness rule: an attempt cap. In practice an agent loop is usually watched by several independent stop signals at once, because no single one is fully trustworthy on its own — a verifier that passes, a hard cap on iterations, and a cost or token budget that can end the run even before the iteration cap is hit. Each iteration is a real cost (here, a stand-in "cost per attempt"), so a loop that only checks "did it pass yet?" can burn a large, unbounded budget before anyone notices.

This notebook stays fully local, like `example.py` — Day 13 is not one of this course's model-backed days, so there is no `ollama` call here. What changes is that `route_after_verify` now checks three conditions instead of one, and we run the same graph three times with different inputs to watch each stop reason happen on its own.

**Graph:** `START -> act -> verify -> (route) -> act / END`

## Setup — declaring the state

`example.py` builds its graph with `StateGraph(dict)` — a plain, unlabeled dictionary. LangGraph then has no idea what fields exist, so it treats the whole state as one opaque blob: whatever a node returns **replaces it entirely**, which is why `example.py`'s `verify` has to re-emit `attempt` and `result` just to keep them.

This notebook instead declares a `State` class listing every field up front, the same pattern Day 12's notebooks use. With a named schema, LangGraph gives each field its **own channel** and merges a node's return **key by key** — so a node only has to return the field(s) it actually changed, and everything else carries forward untouched. That is the difference you'll see below: every node in this notebook is shorter than the equivalent one would be with a plain `dict`.

In [1]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class State(TypedDict):
    attempt: int
    cost: float
    cost_per_attempt: float
    budget: float
    max_attempts: int
    threshold: int
    result: int
    passed: bool


## `act` — do one unit of work

Increases `attempt` by one, spends `cost_per_attempt` against the running `cost`, and produces a `result`. Thanks to the `State` schema above, `act` only needs to return the three fields it actually changed: `attempt`, `cost`, and `result`. Everything else — `cost_per_attempt`, `budget`, `max_attempts`, `threshold` — carries forward on its own, because each field lives in its own channel.

In [2]:
def act(state: State):
    attempt = state["attempt"] + 1
    cost = state["cost"] + state["cost_per_attempt"]
    result = attempt * 5
    return {"attempt": attempt, "cost": cost, "result": result}


## `verify` — check this attempt, nothing else

Same job as `example.py`'s `verify`: it only judges whether *this* `result` clears `threshold`. It has no opinion about cost or attempt count — those belong to the harness logic in the router below. It returns only `passed`; `attempt`, `cost`, `budget`, and everything else are still sitting in their own channels from before, untouched.

In [3]:
def verify(state: State):
    return {"passed": state["result"] >= state["threshold"]}


## `route_after_verify` — the harness, with three stop signals

This is where loop engineering happens. The router reads `passed` (the loop's own verdict) plus two harness-imposed limits it did not decide itself: `budget` (a resource cap) and `max_attempts` (an iteration cap). Any one of the three can end the run; none of them can override the others. Compare this to `example.py`'s `enforce_limit`, which only had the attempt cap — this is the same idea, extended to match the "several stop signals at once" pattern real agent loops need, since a model's (or here, a process's) own sense of "I'm done" is not something you can fully trust alone.

In [4]:
def route_after_verify(state: State):
    if state["passed"]:
        return "done"
    if state["cost"] >= state["budget"]:
        return "done"
    if state["attempt"] >= state["max_attempts"]:
        return "done"
    return "retry"


## Build the graph

In [5]:
graph_builder = StateGraph(State)
graph_builder.add_node("act", act)
graph_builder.add_node("verify", verify)
graph_builder.add_edge(START, "act")
graph_builder.add_edge("act", "verify")
graph_builder.add_conditional_edges(
    "verify", route_after_verify, {"retry": "act", "done": END}
)
graph = graph_builder.compile()


## Run it three times

`stop_reason` is plain Python that reads the final state after `graph.invoke` returns — exactly like `example.py`'s closing `if`/`else` — to say in words which of the three signals ended the run. Watch all three happen across the three configurations below: Run A reaches `threshold` and passes; Run B has an unreachable `threshold` but a generous `budget`, so the iteration cap ends it first; Run C has the same unreachable `threshold` but a tight `budget`, so the cost cap ends it *before* the iteration cap would have.

In [6]:
def stop_reason(result):
    if result["passed"]:
        return "verification passed"
    if result["cost"] >= result["budget"]:
        return f"cost budget ({result['budget']}) reached"
    return f"iteration cap ({result['max_attempts']}) reached"


def run(label, initial_state):
    result = graph.invoke(initial_state)
    print(
        f"{label}: attempt={result['attempt']} cost={result['cost']} "
        f"result={result['result']} -> stopped because {stop_reason(result)}"
    )


run("Run A (reachable threshold)", {
    "attempt": 0, "cost": 0.0, "cost_per_attempt": 1.0,
    "budget": 100.0, "max_attempts": 5, "threshold": 10,
})
run("Run B (unreachable threshold, room in budget)", {
    "attempt": 0, "cost": 0.0, "cost_per_attempt": 1.0,
    "budget": 100.0, "max_attempts": 3, "threshold": 1000,
})
run("Run C (unreachable threshold, tight budget)", {
    "attempt": 0, "cost": 0.0, "cost_per_attempt": 10.0,
    "budget": 25.0, "max_attempts": 10, "threshold": 1000,
})


Run A (reachable threshold): attempt=2 cost=2.0 result=10 -> stopped because verification passed
Run B (unreachable threshold, room in budget): attempt=3 cost=3.0 result=15 -> stopped because iteration cap (3) reached
Run C (unreachable threshold, tight budget): attempt=3 cost=30.0 result=15 -> stopped because cost budget (25.0) reached
